In [2]:
from local.constants import WORKSPACE_ROOT
import json

with open(WORKSPACE_ROOT/"secrets/fir.json") as j:
    secrets = json.load(j)

In [ ]:
import paramiko
from paramiko_jump import MultiFactorAuthHandler
from paramiko_jump import SSHJumpClient, MultiFactorAuthHandler

import logging
import sys

# Set up logging to output to the console
logging.basicConfig(
    stream=sys.stdout,
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)

# Specifically set the paramiko logger to DEBUG level
logging.getLogger("paramiko").setLevel(logging.DEBUG)

auth_handler = MultiFactorAuthHandler()
auth_handler.add(secrets["password"])  # For SSH password
auth_handler.add("1")              # For 2FA phone selection

# Connect through jumphost
with SSHJumpClient(auth_handler=auth_handler) as client:
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
    # Connect to jumphost
    client.connect(
        hostname="localhost",
        port=2222,
        username="root",
        key_filename="~/.ssh/vpn",
        look_for_keys=False,
        allow_agent=False
    )
    
    # # Open tunnel to target host
    # target_transport = client.open_channel_to_host(
    #     host=secrets["host"],
    #     port=22,
    #     timeout=30
    # )
    
    # # Create new SSH session on target
    # target_client = paramiko.SSHClient()
    # target_client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    # target_client.connect(
    #     socket=target_transport,
    #     username=secrets["username"],
    #     password=secrets["password"],  # or use key-based auth
    #     look_for_keys=False,
    #     allow_agent=False,
    # )
    
    # # Execute command on target
    # stdin, stdout, stderr = target_client.exec_command("ls -la")
    # print(stdout.read().decode())

2025-12-01 22:11:42,062 - paramiko.transport - DEBUG - starting thread (client mode): 0x384306e0
2025-12-01 22:11:42,064 - paramiko.transport - DEBUG - Local version/idstring: SSH-2.0-paramiko_4.0.0
2025-12-01 22:11:42,068 - paramiko.transport - DEBUG - Remote version/idstring: SSH-2.0-OpenSSH_9.7
2025-12-01 22:11:42,069 - paramiko.transport - INFO - Connected (version 2.0, client OpenSSH_9.7)
2025-12-01 22:11:42,080 - paramiko.transport - DEBUG - === Key exchange possibilities ===
2025-12-01 22:11:42,081 - paramiko.transport - DEBUG - kex algos: sntrup761x25519-sha512@openssh.com, curve25519-sha256, curve25519-sha256@libssh.org, ecdh-sha2-nistp256, ecdh-sha2-nistp384, ecdh-sha2-nistp521, diffie-hellman-group-exchange-sha256, diffie-hellman-group16-sha512, diffie-hellman-group18-sha512, diffie-hellman-group14-sha256, ext-info-s, kex-strict-s-v00@openssh.com
2025-12-01 22:11:42,082 - paramiko.transport - DEBUG - server key: rsa-sha2-512, rsa-sha2-256
2025-12-01 22:11:42,083 - paramiko.t

AuthenticationException: Authentication failed.

2025-12-01 22:11:42,257 - paramiko.transport - DEBUG - EOF in transport thread


In [9]:
# Now you can run commands, for example:
stdin, stdout, stderr = client.exec_command('ls -la')
print(stdout.read().decode())

2025-12-01 21:54:14,253 - paramiko.transport - DEBUG - [chan 0] Max packet in: 32768 bytes
2025-12-01 21:54:14,267 - paramiko.transport - WARNING - Oops, unhandled type 3 ('unimplemented')


KeyboardInterrupt: 

In [ ]:
client.close()